## 세포라 단일 상품 리뷰데이터 추출코드

In [ ]:
import requests
import pandas as pd
import time
import json

def get_sephora_reviews(product_id, max_pages=5):
    all_reviews = []
    limit = 100  # API 효율을 위해 100으로 설정
    passkey = "calXm2DyQVjcCy9agq85vmTJv5ELuuBCF2sdg4BnJzJus"
    
    for page in range(max_pages):
        offset = page * limit
        url = "https://api.bazaarvoice.com/data/reviews.json"
        
        params = {
            "Filter": [f"contentlocale:en*", f"ProductId:{product_id}"],
            "Sort": "SubmissionTime:desc",
            "Limit": limit,
            "Offset": offset,
            "Include": "Products,Comments",
            "Stats": "Reviews",
            "passkey": passkey,
            "apiversion": "5.4",
            "Locale": "en_US"
        }
        
        try:
            response = requests.get(url, params=params)
            data = response.json()
            
            reviews = data.get('Results', [])
            
            if not reviews:
                print(f"\n✅ 모든 리뷰 수집 완료 (총 {len(all_reviews)}개)")
                break
                
            for rev in reviews:
                # 데이터 추출
                review_data = {
                    "ReviewId": rev.get("Id"),
                    "Author": rev.get("UserNickname"),
                    "Rating": rev.get("Rating"),
                    "Title": rev.get("Title"),
                    "ReviewText": rev.get("ReviewText"),
                    "SubmissionTime": rev.get("SubmissionTime"),
                    "IsRecommended": rev.get("IsRecommended"),
                    # 인센티브 여부 안전하게 추출
                    "Incentivized": rev.get("ContextDataValues", {}).get("IncentivizedReview", {}).get("ValueLabel", "N/A"),
                    "Helpfulness": rev.get("Helpfulness")
                }
                all_reviews.append(review_data)
            
            print(f"Page {page+1} 수집 중... (현재 {len(all_reviews)}개)", end='\r')
            time.sleep(0.5)
            
        except Exception as e:
            print(f"\n❌ 에러 발생: {e}")
            break
            
    # --- JSON 파일 저장 (리스트 형태 직접 저장) ---
    filename = f"sephora_test_crawling.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_reviews, f, ensure_ascii=False, indent=4)
    print(f"\n📂 '{filename}' 저장 완료!")
    
    return pd.DataFrame(all_reviews)

# --- 실행 부분 ---
PRODUCT_ID = "P514736"
df = get_sephora_reviews(PRODUCT_ID, max_pages=10)

# 데이터프레임 상위 5개 확인
if not df.empty:
    display(df.head())

## 세포라 상품 top 10

In [1]:
import requests
import json
from datetime import datetime

API_URL = (
    "https://www.sephora.com/api/v2/catalog/categories/skincare/seo"
    "?targetSearchEngine=NLP"
    "&sortBy=P_BEST_SELLING%3A1%3A%3AP_RATING%3A1%3A%3AP_PROD_NAME%3A0"
    "&currentPage=1"
    "&pageSize=60"
    "&content=true"
    "&includeRegionsMap=true"
    "&pickupRampup=true"
    "&sddRampup=true"
    "&includeEDD=true"
    "&loc=en-US"
    "&ch=rwd"
    "&user-segment=external-app"
)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.sephora.com/shop/skincare?sortBy=BEST_SELLING",
    "Origin": "https://www.sephora.com",
}

print('📡 Sephora API 호출 중...')
response = requests.get(API_URL, headers=HEADERS, timeout=20)
response.raise_for_status()
data = response.json()
print(f'✅ 응답 성공! (Status: {response.status_code})')

products_raw = (
    data.get('products')
    or data.get('catalog', {}).get('products')
    or []
)

if not products_raw:
    print('⚠️ 상품 목록을 찾지 못했습니다. 최상위 키:', list(data.keys()))
    with open('sephora_raw_response.json', 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print('🔍 sephora_raw_response.json 저장됨 (구조 확인용)')
else:
    final_rankings = []
    for product in products_raw:
        if len(final_rankings) >= 10:
            break
        if (
            product.get('isSponsored')
            or product.get('sponsored')
            or product.get('adBadge')
            or 'sponsored' in str(product.get('badges', [])).lower()
        ):
            continue

        brand        = product.get('brandName', 'N/A')
        title        = product.get('displayName', 'N/A')
        sku          = product.get('currentSku', {})
        price        = sku.get('listPrice') or sku.get('salePrice') or product.get('listPrice') or 'N/A'
        rating       = float(product.get('rating', 0) or 0)
        reviews      = int(product.get('reviews', 0) or 0)
        product_id   = product.get('productId', 'N/A')
        url_path     = product.get('targetUrl') or product.get('url', '')
        product_url  = ('https://www.sephora.com' + url_path) if url_path and not url_path.startswith('http') else (url_path or 'N/A')

        final_rankings.append({
            'rank': len(final_rankings) + 1,
            'brand': brand,
            'title': title,
            'rating': rating,
            'reviews': reviews,
            'price': price,
            'url': product_url,
            'product_id': product_id,
            'platform': 'Sephora',
            'collected_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        })

    filename = f"sephora_skincare_top10_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(final_rankings, f, ensure_ascii=False, indent=4)

    print()
    print('=' * 65)
    print('🏆 세포라 스킨케어 베스트셀러 Top 10 (광고 제외)')
    print('=' * 65)
    for item in final_rankings:
        print(f"{item['rank']:>2}위 | {item['brand']:<20} | {item['title'][:28]:<28} | ⭐{item['rating']} | {item['price']}")
    print('=' * 65)
    print(f'💾 저장 완료: {filename}')

📡 Sephora API 호출 중...
✅ 응답 성공! (Status: 200)

🏆 세포라 스킨케어 베스트셀러 Top 10 (광고 제외)
 1위 | rhode                | Glazing Milk Ceramide Facial | ⭐3.8483 | $20.00 - $32.00
 2위 | rhode                | Peptide Lip Tint Nourishing  | ⭐3.8139 | $20.00
 3위 | The Ordinary         | Niacinamide 10% + Zinc 1%  S | ⭐4.2296 | $6.00 - $10.80
 4위 | Beauty of Joseon     | Day Dew Sunscreen Lightweigh | ⭐4.0385 | $18.00
 5위 | Biodance             | Bio Collagen Real Deep Mask  | ⭐4.2173 | $5.00 - $19.00
 6위 | EADEM                | Le Chouchou Exfoliating + So | ⭐4.4868 | $24.00
 7위 | The Ordinary         | Glycolic Acid 7% Exfoliating | ⭐4.3535 | $9.00 - $13.50
 8위 | Touchland            | Power Mist Hydrating Hand Sa | ⭐4.322 | $10.00 - $12.00
 9위 | Tower 28 Beauty      | SOS Daily Hypochlorous Acid  | ⭐4.0623 | $12.00 - $68.00
10위 | rhode                | Glazing Mist Hydrating Face  | ⭐3.9416 | $30.00
💾 저장 완료: sephora_skincare_top10_20260317_165523.json


## 세포라 top10 수집및 리뷰 수집 코드

In [3]:
import requests
import pandas as pd
import json
import time
import random
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
BV_PASSKEY       = "calXm2DyQVjcCy9agq85vmTJv5ELuuBCF2sdg4BnJzJus"
BV_API_URL       = "https://api.bazaarvoice.com/data/reviews.json"
RANK_SAVE_FILE   = "sephora_rankings_current.jsonl"
REVIEW_SAVE_FILE = "sephora_reviews_master.json"
TARGET_RANK      = 4      # ← 리뷰 수집할 순위

SEPHORA_API_URL = (
    "https://www.sephora.com/api/v2/catalog/categories/skincare/seo"
    "?targetSearchEngine=NLP"
    "&sortBy=P_BEST_SELLING%3A1%3A%3AP_RATING%3A1%3A%3AP_PROD_NAME%3A0"
    "&currentPage=1&pageSize=60&content=true"
    "&includeRegionsMap=true&pickupRampup=true&sddRampup=true"
    "&includeEDD=true&loc=en-US&ch=rwd&user-segment=external-app"
)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.sephora.com/shop/skincare?sortBy=BEST_SELLING",
    "Origin": "https://www.sephora.com",
}


# ── 함수: 리뷰 전체 수집 (페이지 제한 없이 끝까지) ───────────────
def get_sephora_reviews_all(product_id, product_name):
    all_reviews = []
    limit  = 100
    offset = 0
    total  = None   # 첫 응답에서 TotalResults 확인

    print(f"\n🚀 [{product_name}] 전체 리뷰 수집 시작...")

    while True:
        params = {
            "Filter"    : ["contentlocale:en*", f"ProductId:{product_id}"],
            "Sort"      : "SubmissionTime:desc",
            "Limit"     : limit,
            "Offset"    : offset,
            "Include"   : "Products,Comments",
            "Stats"     : "Reviews",
            "passkey"   : BV_PASSKEY,
            "apiversion": "5.4",
            "Locale"    : "en_US",
        }
        try:
            response = requests.get(BV_API_URL, params=params, timeout=15)
            data = response.json()

            # 첫 페이지에서 전체 리뷰 수 확인
            if total is None:
                total = data.get("TotalResults", 0)
                print(f"📊 수집 대상 전체 리뷰 수: {total}개")

            reviews = data.get("Results", [])
            if not reviews:
                print(f"\n✅ 수집 완료 (총 {len(all_reviews)}개 / 전체 {total}개)")
                break

            for rev in reviews:
                all_reviews.append({
                    "product_id"    : product_id,
                    "ReviewId"      : rev.get("Id"),
                    "Author"        : rev.get("UserNickname"),
                    "Rating"        : rev.get("Rating"),
                    "Title"         : rev.get("Title"),
                    "ReviewText"    : rev.get("ReviewText"),
                    "SubmissionTime": rev.get("SubmissionTime"),
                    "IsRecommended" : rev.get("IsRecommended"),
                    "Incentivized"  : rev.get("ContextDataValues", {}).get("IncentivizedReview", {}).get("ValueLabel", "N/A"),
                    "Helpfulness"   : rev.get("Helpfulness"),
                })

            offset += limit
            print(f"🔄 수집 중... {len(all_reviews)} / {total}개", end="\r")
            time.sleep(random.uniform(0.4, 0.7))

            # 전체 수집 완료 체크
            if total and len(all_reviews) >= total:
                print(f"\n✅ 수집 완료 (총 {len(all_reviews)}개 / 전체 {total}개)")
                break

        except Exception as e:
            print(f"\n❌ 에러 발생: {e}")
            break

    return all_reviews


# ── STEP 1: 세포라 스킨케어 베스트셀러 Top 10 수집 ────────────────
print("📡 Sephora API 호출 중...")
resp = requests.get(SEPHORA_API_URL, headers=HEADERS, timeout=20)
resp.raise_for_status()
data = resp.json()
print(f"✅ 응답 성공! (Status: {resp.status_code})")

products_raw = (
    data.get("products")
    or data.get("catalog", {}).get("products")
    or []
)

if not products_raw:
    print("⚠️ 상품 목록을 찾지 못했습니다. 최상위 키:", list(data.keys()))
    with open("sephora_raw_response.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print("🔍 sephora_raw_response.json 저장됨 (구조 확인용)")

else:
    rank_data_list = []
    review_master  = []
    rank_count     = 1

    for product in products_raw:
        if rank_count > 10:
            break

        # 광고 제외
        if (
            product.get("isSponsored")
            or product.get("sponsored")
            or product.get("adBadge")
            or "sponsored" in str(product.get("badges", [])).lower()
        ):
            continue

        brand        = product.get("brandName", "N/A")
        title        = product.get("displayName", "N/A")
        sku          = product.get("currentSku", {})
        price        = sku.get("listPrice") or sku.get("salePrice") or product.get("listPrice") or "N/A"
        rating       = float(product.get("rating", 0) or 0)
        reviews_cnt  = int(product.get("reviews", 0) or 0)
        product_id   = product.get("productId", "N/A")
        url_path     = product.get("targetUrl") or product.get("url", "")
        product_url  = ("https://www.sephora.com" + url_path) if url_path and not url_path.startswith("http") else (url_path or "N/A")

        rank_data_list.append({
            "rank"        : rank_count,
            "brand"       : brand,
            "title"       : title,
            "rating"      : rating,
            "reviews"     : reviews_cnt,
            "price"       : price,
            "url"         : product_url,
            "product_id"  : product_id,
            "platform"    : "Sephora",
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })
        print(f"📍 {rank_count}위 확인: [{brand}] {title[:40]}")

        # ── STEP 2: TARGET_RANK 상품 리뷰 전체 수집 ───────────────
        if rank_count == TARGET_RANK:
            review_master = get_sephora_reviews_all(product_id, title)

        rank_count += 1

    # ── STEP 3: 저장 ──────────────────────────────────────────────
    with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
        for entry in rank_data_list:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump(review_master, f, ensure_ascii=False, indent=4)

    # ── STEP 4: 결과 출력 ─────────────────────────────────────────
    print()
    print("=" * 65)
    print("🏆 세포라 스킨케어 베스트셀러 Top 10 (광고 제외)")
    print("=" * 65)
    for item in rank_data_list:
        marker = " ◀ 리뷰 수집됨" if item["rank"] == TARGET_RANK else ""
        print(f"{item['rank']:>2}위 | {item['brand']:<20} | {item['title'][:28]:<28} | ⭐{item['rating']} | {item['price']}{marker}")
    print("=" * 65)
    print(f"\n📂 저장 완료")
    print(f"  - 순위 파일 : {RANK_SAVE_FILE}")
    print(f"  - 리뷰 파일 : {REVIEW_SAVE_FILE} (총 {len(review_master)}개)")

    # ── STEP 5: DataFrame 확인 ────────────────────────────────────
    if review_master:
        df = pd.DataFrame(review_master)
        print(f"\n📋 {TARGET_RANK}위 상품 리뷰 상위 5개")
        display(df.head())

📡 Sephora API 호출 중...
✅ 응답 성공! (Status: 200)
📍 1위 확인: [rhode] Glazing Milk Ceramide Facial Essence
📍 2위 확인: [rhode] Peptide Lip Tint Nourishing Glaze
📍 3위 확인: [The Ordinary] Niacinamide 10% + Zinc 1%  Serum for Oil
📍 4위 확인: [Beauty of Joseon] Day Dew Sunscreen Lightweight SPF 50

🚀 [Day Dew Sunscreen Lightweight SPF 50] 전체 리뷰 수집 시작...
📊 수집 대상 전체 리뷰 수: 339개
🔄 수집 중... 339 / 339개
✅ 수집 완료 (총 339개 / 전체 339개)
📍 5위 확인: [Biodance] Bio Collagen Real Deep Mask for Pore Min
📍 6위 확인: [EADEM] Le Chouchou Exfoliating + Softening Pept
📍 7위 확인: [The Ordinary] Glycolic Acid 7% Exfoliating and Brighte
📍 8위 확인: [Touchland] Power Mist Hydrating Hand Sanitizer
📍 9위 확인: [Tower 28 Beauty] SOS Daily Hypochlorous Acid Spray for Br
📍 10위 확인: [rhode] Glazing Mist Hydrating Face Spray

🏆 세포라 스킨케어 베스트셀러 Top 10 (광고 제외)
 1위 | rhode                | Glazing Milk Ceramide Facial | ⭐3.8483 | $20.00 - $32.00
 2위 | rhode                | Peptide Lip Tint Nourishing  | ⭐3.8139 | $20.00
 3위 | The Ordinary         | Niacina

,product_id,ReviewId,Author,Rating,Title,ReviewText,SubmissionTime,IsRecommended,Incentivized,Helpfulness
0,P517678,381862495,NataliaRo,5,NOT Greasy but definitely DEWY,This definitely does what it says it’s going t...,2026-03-17T03:47:08.000+00:00,True,No,NaN
1,P517678,381467560,ZTriv,1,Greasy not dewy,It’s greasy. Doesn’t leave white cast. But I g...,2026-03-12T02:54:15.000+00:00,False,No,0.4
2,P517678,381427197,vickysnoop,4,"Good, inexpensive sunscreen","It’s good, when taking off it burns/hurts eyes...",2026-03-11T18:23:42.000+00:00,True,N/A,1.0
3,P517678,381381019,deeeee565,5,The best sunscreen,"It is a great size , and it is fees light on m...",2026-03-11T01:23:09.000+00:00,True,No,1.0
4,P517678,381042578,kimper,5,Great SPF that dubs as a primer,I was at the Stonestown (California) store pic...,2026-03-07T21:45:17.000+00:00,True,No,1.0
